# Algorithm 4.2 — State vector from the orbital elements

**Goal:** the inverse of 4.1. Given the six classical elements $(h, e, \Omega, i, \omega, \theta)$, rebuild the position $\mathbf{r}$ and velocity $\mathbf{v}$ in the geocentric-equatorial frame.

**Code (answer key):** [`sv_from_coe`](../curtis/algorithms/alg_4_2_sv_from_coe.py) · **Book:** §4.4, Algorithm 4.2 · **Worked example:** 4.5

## Read first

| Symbol | Name | Equation |
|---|---|---|
| $\mathbf{r}_p,\ \mathbf{v}_p$ | state in the orbit's own (perifocal) frame | 4.37, 4.38 |
| $R_3(\Omega),\ R_1(i),\ R_3(\omega)$ | elementary rotations about $z, x, z$ | 4.39–4.41 |
| $[Q]_{\bar X p}$ | perifocal → geocentric-equatorial matrix | 4.44 |
| $\mathbf{r},\ \mathbf{v}$ | state in the geocentric-equatorial frame | 4.46 |

The whole algorithm is two moves: **write the state where the orbit is simple, then rotate it into the frame you want.**

## The picture (the two-stage idea)

In 4.1 you *projected* $\mathbf{r},\mathbf{v}$ onto the frame to read off the angles. Here you run it backwards.

1. **Perifocal frame** — attach axes to the orbit itself: $\hat{\mathbf{p}}$ toward perigee, $\hat{\mathbf{q}}$ a quarter-turn ahead in the orbit plane, $\hat{\mathbf{w}}$ along $\mathbf{h}$. In *these* axes the orbit lies flat in the $p$–$q$ plane and the state is just the orbit equation — no angles needed.
2. **Rotate** that perifocal state into the geocentric-equatorial frame using the three orientation angles $\Omega, i, \omega$.

So $h$ and $e$ and $\theta$ set the **size/shape/where-on-the-orbit** (stage 1); $\Omega, i, \omega$ set **how the orbit is tilted in space** (stage 2).

In [ ]:
import numpy as np
import orbital_elements

In [ ]:
# Reveal the elements one at a time (uncomment to run):
# for f in ("frame", "raan", "inclination", "argp", "ta"):
#     orbital_elements.plot_orbital_elements(focus=f).show()

In [ ]:
# The frame, the orbit, and all six elements together — drag to rotate.
orbital_elements.plot_orbital_elements(focus="all").show()

## See it — what the inputs mean

The six elements are exactly the labelled quantities in the figure above. 4.2 reads them **as inputs** and reconstructs the picture:

- $h, e$ → the **size and shape** of the orbit (via $r = \tfrac{h^2/\mu}{1+e\cos\theta}$).
- $\theta$ → **where on the orbit** the satellite sits (measured from perigee).
- $\Omega, i, \omega$ → **how the orbit plane is oriented** in the geocentric-equatorial frame: swing the node by $\Omega$, tilt by $i$, then turn perigee by $\omega$.

Stage 1 builds the state in the orbit's own (perifocal) axes; stage 2 is the rotation that carries it into the $\hat{\mathbf{I}}\hat{\mathbf{J}}\hat{\mathbf{K}}$ frame you see.

## Where these equations come from

### 1 · The perifocal frame, where the orbit is trivial
With $\hat{\mathbf{p}}$ toward perigee and $\hat{\mathbf{q}}$ $90°$ ahead, the position is $\mathbf{r}=r\cos\theta\,\hat{\mathbf{p}}+r\sin\theta\,\hat{\mathbf{q}}$. Substituting the orbit equation $r=\dfrac{h^2/\mu}{1+e\cos\theta}$ (derived in 4.1) gives
$$\mathbf{r}_p=\frac{h^2}{\mu}\,\frac{1}{1+e\cos\theta}\begin{bmatrix}\cos\theta\\[2pt]\sin\theta\\[2pt]0\end{bmatrix}\qquad(4.37).$$
Differentiate $\mathbf{r}_p$ in time, using $\dot\theta=h/r^2$ and $\dot r=\tfrac{\mu}{h}e\sin\theta$. The $r$-dependence cancels cleanly and leaves
$$\mathbf{v}_p=\frac{\mu}{h}\begin{bmatrix}-\sin\theta\\[2pt]e+\cos\theta\\[2pt]0\end{bmatrix}\qquad(4.38).$$
Both lie in the $p$–$q$ plane (third component $0$) — that is the point of this frame.

### 2 · The rotation into the geocentric-equatorial frame
To carry a vector **from** the geocentric frame **into** the perifocal frame, undo the three orientation angles in order — spin by $\Omega$ about $\hat{\mathbf{K}}$ to reach the node, tilt by $i$ about the node line, then spin by $\omega$ about $\hat{\mathbf{w}}$ to reach perigee:
$$[Q]_{p\bar X}=R_3(\omega)\,R_1(i)\,R_3(\Omega),$$
with the elementary matrices
$$R_3(\phi)=\begin{bmatrix}\cos\phi&\sin\phi&0\\-\sin\phi&\cos\phi&0\\0&0&1\end{bmatrix},\quad R_1(\phi)=\begin{bmatrix}1&0&0\\0&\cos\phi&\sin\phi\\0&-\sin\phi&\cos\phi\end{bmatrix}.$$
We want the **opposite** direction (perifocal → geocentric), so transpose (for a rotation, the inverse *is* the transpose):
$$[Q]_{\bar X p}=\big(R_3(\omega)\,R_1(i)\,R_3(\Omega)\big)^{T}\qquad(4.44).$$
Then
$$\mathbf{r}=[Q]_{\bar X p}\,\mathbf{r}_p,\qquad \mathbf{v}=[Q]_{\bar X p}\,\mathbf{v}_p\qquad(4.46).$$
Rotation matrices are **orthogonal**, so this preserves every length and angle — $|\mathbf{r}|$, $|\mathbf{v}|$, $h$ and the energy are unchanged; only the *components* (the axes they are written in) change.

## Step by step (in code order)

**1. Unpack** the elements: $h, e, \Omega, i, \omega, \theta$ (angles already in **radians**).

**2. Perifocal position (Eq 4.37).** $\;\mathbf{r}_p=\dfrac{h^2}{\mu}\dfrac{1}{1+e\cos\theta}\,[\cos\theta,\ \sin\theta,\ 0]$

**3. Perifocal velocity (Eq 4.38).** $\;\mathbf{v}_p=\dfrac{\mu}{h}\,[-\sin\theta,\ e+\cos\theta,\ 0]$

**4. Elementary rotations (Eqs 4.39–4.41).** Build $R_3(\Omega),\ R_1(i),\ R_3(\omega)$.

**5. Transformation matrix (Eq 4.44).** $\;[Q]_{\bar X p}=\big(R_3(\omega)\,R_1(i)\,R_3(\Omega)\big)^{T}$

**6. Rotate to the geocentric frame (Eq 4.46).** $\;\mathbf{r}=[Q]_{\bar X p}\,\mathbf{r}_p,\quad \mathbf{v}=[Q]_{\bar X p}\,\mathbf{v}_p$

**↓ Now type your implementation below.** No quadrant fixes this time — it is a direct construction. The module linked above is your answer key; peek only *after* you have tried. (`@` is NumPy's matrix multiply; `.T` transposes.)

In [ ]:
def sv_from_coe(coe, mu):
    """State vector (r, v) from classical elements [h, e, RA, incl, w, TA] (radians)."""
    h, e, RA, incl, w, TA = coe

    # 2. Perifocal position  rp = (h^2/mu)/(1 + e cos TA) * [cos TA, sin TA, 0]   (Eq 4.37)

    # 3. Perifocal velocity  vp = (mu/h) * [-sin TA, e + cos TA, 0]               (Eq 4.38)

    # 4. Elementary rotations R3(RA), R1(incl), R3(w)                             (Eqs 4.39-4.41)
    #    R3(p) = [[ cos p, sin p, 0], [-sin p, cos p, 0], [0, 0, 1]]
    #    R1(p) = [[1, 0, 0], [0, cos p, sin p], [0, -sin p, cos p]]

    # 5. Perifocal -> geocentric matrix  Q = (R3(w) @ R1(incl) @ R3(RA)).T        (Eq 4.44)

    # 6. r = Q @ rp ;  v = Q @ vp                                                 (Eq 4.46)

    # return r, v
    raise NotImplementedError("fill in steps 2-6, then delete this line")

## Verify — Example 4.5

**Input** ($\mu_\oplus=398600$): $\;h=80000$ km²/s, $\;e=1.4$, $\;\Omega=40°$, $\;i=30°$, $\;\omega=60°$, $\;\theta=30°$.

(Note $e=1.4>1$ — this is a **hyperbola**.)

| | $x$ | $y$ | $z$ |
|---|---|---|---|
| $\mathbf{r}$ (km) | −4039.9 | 4814.56 | 3628.62 |
| $\mathbf{v}$ (km/s) | −10.386 | −4.77192 | 1.74388 |

Run the cell below once your function is typed.

In [ ]:
deg = np.pi / 180
mu  = 398600.0
coe = [80000.0, 1.4, 40*deg, 30*deg, 60*deg, 30*deg]   # [h, e, RA, incl, w, TA]

r, v = sv_from_coe(coe, mu)

print(f"r (km)   = [{r[0]:10.5g} {r[1]:10.5g} {r[2]:10.5g}]   (expect [-4039.9  4814.56  3628.62])")
print(f"v (km/s) = [{v[0]:10.5g} {v[1]:10.5g} {v[2]:10.5g}]   (expect [-10.386  -4.77192  1.74388])")

assert np.allclose(r, [-4039.9, 4814.56, 3628.62], atol=1e-2)
assert np.allclose(v, [-10.386, -4.77192, 1.74388], atol=1e-4)
print("\nAll checks passed ✔")

In [ ]:
# The round trip: 4.2 then 4.1 should return the inputs you started with.
import sys; sys.path.insert(0, "..")
from curtis.algorithms.alg_4_1_coe_from_sv import coe_from_sv   # answer-key 4.1

back = coe_from_sv(r, v, mu)            # state -> elements
print(f"e    = {back[1]:.6g}      (expect 1.4)")
print(f"RA   = {np.degrees(back[2]):.5g} deg   (expect 40)")
print(f"incl = {np.degrees(back[3]):.5g} deg   (expect 30)")
print(f"w    = {np.degrees(back[4]):.5g} deg   (expect 60)")
print(f"TA   = {np.degrees(back[5]):.5g} deg   (expect 30)")

assert abs(back[1] - 1.4) < 1e-6
assert abs(np.degrees(back[2]) - 40) < 1e-3
assert abs(np.degrees(back[5]) - 30) < 1e-3
print("\nRound trip 4.5 -> state -> 4.5 recovered ✔")

## What this confirms

- The trick of orbital mechanics: **solve in the easy frame, then rotate.** Stage 1 (perifocal) needs no angles; stage 2 is a pure rotation.
- Rotation matrices are **orthogonal** ($Q^{-1}=Q^{T}$), so the transform preserves $|\mathbf{r}|$, $|\mathbf{v}|$, $h$ and the energy — it only changes the axes.
- **4.1 and 4.2 are exact inverses:** the round trip returned the inputs to all printed digits.
- $e=1.4>1$ ⇒ a **hyperbola** ($a<0$): the same construction handles open orbits with no change.

**Next:** Algorithm 3.4 (`rv_from_r0v0`) — propagate a state forward in time, the first algorithm where *time* finally enters.